In [49]:
import oracledb
import pandas as pd
import numpy as np
import re
from IPython.display import display
from datetime import datetime
import hashlib

In [50]:
# Configura tus credenciales y detalles de conexión
usuario = "evaluacion2" 
contrasena = "123456" 
dsn_tns = "localhost/xe"
# Establece la conexión a la base de datos
try: 
    conexion = oracledb.connect(user=usuario, password=contrasena, dsn=dsn_tns)
    print("¡Conexión exitosa a la base de datos Oracle!")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a la base de datos Oracle!


In [ ]:
# Extraemos los datos a DataFrames de pandas usando consultas SQL y la conexión establecida
df_clientes = pd.read_sql("SELECT * FROM CLIENTES", con=conexion)

display(df_clientes)

In [ ]:
# Filtramos y mostramos SOLO los registros que tienen nulos en cada DataFrame 
print("CLIENTES CON DATOS FALTANTES (Nulos)")
# 1. Creamos la copia de trabajo partiendo del original limpio (sin procesar aún)
# Asumiendo que tu DataFrame original se llama df_clientes
cliente_diagnostico = df_clientes.copy()
# IDENTIFICACIÓN DE VALORES NULOS
# Definimos las columnas que te interesa monitorear
columnas_interes = [
  'RUT', 'NOMBRE', 'APELLIDOPATERNO', 'APELLIDOMATERNO',
  'FECHAREGISTRO', 'TELEFONO', 'EMAIL', 'DIRECCION'
]
print("--- REPORTE DE DATOS FALTANTES (Valores Nulos Reales) ---")
# .isnull().sum() nos dirá cuántos NaN hay por columna
reporte_nulos = cliente_diagnostico[columnas_interes].isnull().sum()
print(reporte_nulos)
print("\n" + "-"*60)
# 3. VISUALIZACIÓN DE LA MUESTRA "CRUDA"
print("--- MUESTRA DE DATOS ORIGINALES (Sin modificaciones) ---")
# Mostramos las primeras 10 filas. 
# Los valores nulos aparecerán como 'NaN' o 'None' dependiendo del tipo de dato.
display(cliente_diagnostico[columnas_interes].head(10))
# 4.FILTRAR SOLO FILAS CON NULOS
# Esto es muy útil para ver exactamente qué registros están incompletos
print("\n--- EJEMPLOS DE REGISTROS QUE CONTIENEN NULOS ---")
solo_nulos = cliente_diagnostico[cliente_diagnostico.isnull().any(axis=1)]
display(solo_nulos.head(5))

In [ ]:
#  CREAMOS UNA COPIA DE SEGURIDAD 
# Todo el procesamiento lo haremos sobre df_ejercicio para no dañar df_clientes
df_ejercicio = df_clientes.copy()
print("INICIANDO PROCESAMIENTO DE REGLAS DE NEGOCIO SOBRE CLIENTES")
print(f"Total de registros iniciales a evaluar: {len(df_ejercicio)}\n")

In [ ]:
#  CLASIFICACIÓN DE ESTADOS 

# Todos parten como "Sin Cambios" por defecto y luego se irán actualizando según las reglas de negocio
df_ejercicio['ESTADO'] = 'Sin Cambios'

print("Columna ESTADO creada.")
# Vemos los primeros registros para confirmar que la columna ESTADO se creó correctamente y tiene el valor por defecto
display(df_ejercicio[['IDCLIENTE', 'RUT', 'ESTADO']].head(3))

In [ ]:
# MANEJO DE CAMPOS CLAVE NULOS (Descarte inmediato)

# Si RUT o IDCLIENTE están vacíos, se descarta el registro completo y se marca como "Descartado" en ESTADO
mask_nulos_clave = df_ejercicio['RUT'].isna() | df_ejercicio['IDCLIENTE'].isna() | (df_ejercicio['RUT'] == '')
df_ejercicio.loc[mask_nulos_clave, 'ESTADO'] = 'Descartado'

print("Registros con campos clave vacíos descartados.")
# Mostramos específicamente los que fueron descartados para verificar que la regla se aplicó correctamente
display(df_ejercicio[df_ejercicio['ESTADO'] == 'Descartado'][['IDCLIENTE', 'RUT', 'ESTADO']].head(3))

In [ ]:
# ESTANDARIZACIÓN DE ESTRUCTURAS (Nombres separados por comas)

def separar_nombres(row):
    # Solo intentamos separar si hay una coma y el registro no está descartado y el campo NOMBRE no es nulo
    if row['ESTADO'] != 'Descartado' and pd.notna(row['NOMBRE']) and ',' in str(row['NOMBRE']):
        partes = [p.strip() for p in str(row['NOMBRE']).split(',')]
        row['NOMBRE'] = partes[0] # El primer elemento es el Nombre
        
        if len(partes) > 1:
            row['APELLIDOPATERNO'] = partes[1]
        if len(partes) > 2:
            row['APELLIDOMATERNO'] = partes[2]
            
        row['ESTADO'] = 'Transformados' # Se alteró la estructura del nombre, se marca como Transformado
    return row

df_ejercicio = df_ejercicio.apply(separar_nombres, axis=1)

print("Nombres separados correctamente.")
# Mostramos los primeros 10 registros para ver cómo quedaron distribuidos los nombres y los estados después de la transformación 
display(df_ejercicio[['IDCLIENTE', 'NOMBRE', 'APELLIDOPATERNO', 'APELLIDOMATERNO', 'ESTADO']].head(10))

In [ ]:
# MANEJO DE CAMPOS OPCIONALES INCOMPLETOS

campos_opcionales = ['APELLIDOPATERNO', 'APELLIDOMATERNO', 'DIRECCION', 'CIUDAD', 'CODIGOPOSTAL', 'TELEFONO', 'EMAIL']

for col in campos_opcionales:
    # Identificamos valores nulos o vacíos
    mask_nulo = df_ejercicio[col].isna() | (df_ejercicio[col] == '')
    
    # Asignamos el valor predeterminado
    df_ejercicio.loc[mask_nulo, col] = 'NO DISPONIBLE'
    
    # Marcamos como transformado (solo si no estaba ya descartado)
    mask_transformar = mask_nulo & (df_ejercicio['ESTADO'] != 'Descartado')
    df_ejercicio.loc[mask_transformar, 'ESTADO'] = 'Transformados'

print("Campos vacíos rellenados con 'NO DISPONIBLE'.")
# Mostramos cómo se ven ahora los campos que antes tenían NaN
display(df_ejercicio[['IDCLIENTE', 'NOMBRE', 'APELLIDOPATERNO','APELLIDOMATERNO', 'DIRECCION','TELEFONO', 'EMAIL', 'ESTADO']].head(10))

In [ ]:
# ESTANDARIZACIÓN DE FECHAS A FORMATO ISO 8601
def estandarizar_fecha(fecha):
    # Si es nulo o ya lo rellenamos con 'NO DISPONIBLE', lo dejamos igual y no intentamos convertir
    if pd.isna(fecha) or fecha == 'NO DISPONIBLE': return fecha
    
    fecha_str = str(fecha).strip()
    
    # Lista actualizada de formatos (Se agregó '%m%d%Y' para atrapar MMDDYYYY) y '%Y/%m/%d' para atrapar YYYY/MM/DD
    formatos = ['%d/%m/%Y', '%m-%d-%Y', '%Y-%m-%d', '%d-%m-%Y', '%d%m%Y', '%m%d%Y', '%Y/%m/%d']
    
    for fmt in formatos:
        try:
            # Intenta convertir la fecha y, si calza, la devuelve en AAAA-MM-DD usando strftime
            return datetime.strptime(fecha_str, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return fecha # Si no coincide con nada, retorna el original sin cambios (podría ser un formato raro o un error que no queremos modificar)

# Guardamos las fechas originales para comparar después de la transformación y así decidir si marcamos como Transformado o no
fechas_originales = df_ejercicio['FECHAREGISTRO'].copy()

# Aplicamos la función de estandarización a la columna de fechas y actualizamos el estado si hubo un cambio visual en la fecha (y no estaba descartado) 
df_ejercicio['FECHAREGISTRO'] = df_ejercicio['FECHAREGISTRO'].apply(estandarizar_fecha)

# Cambiamos el estado a 'Transformados' si la fecha fue modificada visualmente y el registro no está descartado 
mask_fecha_cambiada = (fechas_originales != df_ejercicio['FECHAREGISTRO']) & (df_ejercicio['ESTADO'] != 'Descartado')
df_ejercicio.loc[mask_fecha_cambiada, 'ESTADO'] = 'Transformados'

print("Fechas estandarizadas a formato AAAA-MM-DD.")
# Mostramos un mix de registros para que veas que ahora todas las fechas lucen iguales y cómo se actualizó el estado de cada uno según 
# si se transformó o no la fecha y si no estaban descartados para que veas el resultado de la transformación de fechas y los estados asociados 
display(df_ejercicio[['IDCLIENTE', 'FECHAREGISTRO', 'ESTADO']].head(10))

In [ ]:
# FORMATEO, VALIDACIÓN Y DESCARTE DE RUT


def formatear_rut(rut):
    if pd.isna(rut) or rut == '': return "XX.XXX.XXX-X"
    
    # Limpiamos todo para dejar solo letras y números y el dígito verificador, y lo ponemos en mayúscula para estandarizar 
    # (esto ayuda a validar mejor y a formatear correctamente) 
    rut_limpio = str(rut).upper().replace(".", "").replace("-", "").strip()
    if len(rut_limpio) < 2: return "XX.XXX.XXX-X"
        
    cuerpo = rut_limpio[:-1]
    dv = rut_limpio[-1]
    
    try:
        int(cuerpo) # Validamos que el cuerpo sea numérico 
        # Formateamos con puntos de miles y guion antes del dígito verificador 
        return f"{int(cuerpo):,}".replace(",", ".") + f"-{dv}"
    except ValueError:
        return "XX.XXX.XXX-X" # Si el cuerpo tiene letras, es inválido (Valor de descarte) 

# Guardamos los RUTs como venían originalmente para comparar después de la transformación y así decidir si marcamos como 
# Transformado o no y para evidenciar los cambios después de la transformación y validación del RUT y que veas cómo quedaron los 
# RUTs después de la transformación y validación y los estados asociados a cada uno según si se transformó o se descartó por el RUT 
ruts_originales = df_ejercicio['RUT'].copy()

# Aplicamos la función de limpieza y formateo a la columna de RUT y actualizamos el estado según corresponda (Descartado si quedó 
# como "XX.XXX.XXX-X" o Transformado si se arregló y es diferente al original y no estaba descartado) 
df_ejercicio['RUT'] = df_ejercicio['RUT'].apply(formatear_rut)

# Si el RUT quedó como "XX.XXX.XXX-X", la regla de negocio dice que se DESCARTA y se marca como "Descartado" en ESTADO 
# (esto incluye tanto los que eran nulos o vacíos como los que tenían un formato tan malo que no se pudo arreglar, como los que tenían letras en el cuerpo del RUT, etc.) 
df_ejercicio.loc[df_ejercicio['RUT'] == 'XX.XXX.XXX-X', 'ESTADO'] = 'Descartado'

# Si el RUT se arregló y es válido, se marca como Transformado si es diferente al original y no estaba descartado 
mask_rut_cambiado = (ruts_originales != df_ejercicio['RUT']) & (df_ejercicio['ESTADO'] != 'Descartado')
df_ejercicio.loc[mask_rut_cambiado, 'ESTADO'] = 'Transformados'

print("RUTs formateados y validados.")

# Mostramos registros transformados y descartados por culpa del RUT para evidenciar los cambios y que veas cómo quedaron los RUTs 
# después de la transformación y validación y los estados asociados a cada uno según si se transformó o se descartó por el RUT
print("Ejemplo de registros procesados en esta etapa:")
display(df_ejercicio[['IDCLIENTE', 'RUT', 'ESTADO']].head(10))

In [ ]:
# AUDITORÍA PREVIA DE DUPLICADOS

# Calculamos la cantidad de RUTs que aparecen más de una vez
total_registros = len(df_ejercicio)
conteo_ruts = df_ejercicio['RUT'].value_counts()
ruts_duplicados = conteo_ruts[conteo_ruts > 1]
mask_duplicados_previa = df_ejercicio.duplicated(subset=['RUT'], keep='first')
total_sobrantes = mask_duplicados_previa.sum()

print("=== AUDITORÍA DE DATOS DUPLICADOS ===")
print(f"Total de registros en el dataset: {total_registros}")
print(f"Cantidad de RUTs que tienen duplicados: {len(ruts_duplicados)}")
print(f"Total de registros 'sobrantes' que serán descartados: {total_sobrantes}")
print("-" * 40)

# Mostramos cuáles son los RUTs más repetidos para tenerlos identificados
if not ruts_duplicados.empty:
    print("RUTs con mayor número de repeticiones:")
    display(ruts_duplicados.head(5))
else:
    print("No se encontraron RUTs duplicados.")

In [ ]:
# ORDENAMIENTO CRONOLÓGICO PARA DUPLICADOS

# Primero, convertimos temporalmente a tipo datetime para ordenar de verdad y así identificar correctamente los registros más antiguos en 
# caso de duplicados por RUT y quedarnos con el más reciente que es el que queremos conservar y marcar como Transformado si es que no estaba 
# descartado y los demás como Descargados por ser duplicados antiguos 
df_ejercicio['FECHA_AUX'] = pd.to_datetime(df_ejercicio['FECHAREGISTRO'], errors='coerce')

# Ordenamos por RUT y por FECHA (de la más reciente a la más antigua) para que los duplicados queden ordenados cronológicamente y así podamos identificar 
# fácilmente cuál es el más reciente (el que queremos conservar) y cuáles son los antiguos (los que queremos marcar como descartados por ser duplicados antiguos) 
df_ejercicio = df_ejercicio.sort_values(by=['RUT', 'FECHA_AUX'], ascending=[True, False])

print("Datos ordenados por fecha para identificar duplicados antiguos.")
# Mostramos un ejemplo de cómo se ven los registros ordenados por RUT y por fecha para que veas que ahora están ordenados cronológicamente y así puedas  
# los estados de cada registro y cómo quedaron los RUTs después de la transformación, validación y ordenamiento. 
display(df_ejercicio[['IDCLIENTE', 'RUT', 'FECHAREGISTRO', 'ESTADO']].head(10))

In [ ]:
# IDENTIFICACIÓN Y DESCARTE DE DUPLICADOS


# Identificamos los duplicados basándonos en el RUT. 
# keep='first' significa que el primero (el más reciente) NO se marca como duplicado.
mask_duplicados = df_ejercicio.duplicated(subset=['RUT'], keep='first')

# Solo marcamos como duplicado si el registro no estaba ya descartado por otra razón
# (Por ejemplo, un RUT que ya era XX.XXX.XXX-X no cuenta como duplicado "valido")
df_ejercicio.loc[mask_duplicados & (df_ejercicio['RUT'] != "XX.XXX.XXX-X"), 'ESTADO'] = 'Descartado'

# Eliminamos la columna auxiliar de fecha que usamos para ordenar
df_ejercicio = df_ejercicio.drop(columns=['FECHA_AUX'])

print("Duplicados antiguos identificados y marcados para descarte.")

# Para demostrarlo en la defensa, mostramos si hay algún RUT que se repita en la lista
# pero que ahora tenga estados distintos (uno 'Sin Cambios' o 'Transformado' y el otro 'Descartado')
display(df_ejercicio[df_ejercicio.duplicated(subset=['RUT'], keep=False)].sort_values('RUT').head(6))

In [72]:
# FUNCIÓN DE CIFRADO ÉTICO (SHA-256)

def cifrar_dato(texto):
    if pd.isna(texto) or texto == 'NO DISPONIBLE' or texto == 'XX.XXX.XXX-X':
        return texto
    
    # Convertimos el texto a bytes y aplicamos el hash SHA-256
    hash_object = hashlib.sha256(str(texto).encode())
    # Retornamos solo los primeros 12 caracteres para que sea manejable (y seguro)
    return hash_object.hexdigest()[:12]

print("Función de cifrado PII lista.")

Función de cifrado PII lista.


In [73]:
# ANONIMIZACIÓN DE DATOS SENSIBLES (PII)


# Solo ciframos los datos de registros que pasaron las reglas de negocio
mask_validos = (df_ejercicio['ESTADO'] != 'Descartado')

# Aplicamos cifrado al RUT y al Email
df_ejercicio.loc[mask_validos, 'RUT_CIFRADO'] = df_ejercicio.loc[mask_validos, 'RUT'].apply(cifrar_dato)
df_ejercicio.loc[mask_validos, 'EMAIL_CIFRADO'] = df_ejercicio.loc[mask_validos, 'EMAIL'].apply(cifrar_dato)

# Para los descartados, mantenemos el valor original o el error para auditoría
df_ejercicio.loc[~mask_validos, 'RUT_CIFRADO'] = df_ejercicio.loc[~mask_validos, 'RUT']
df_ejercicio.loc[~mask_validos, 'EMAIL_CIFRADO'] = df_ejercicio.loc[~mask_validos, 'EMAIL']

print("Datos sensibles anonimizados.")
# Mostramos la diferencia entre el dato original y el cifrado
display(df_ejercicio[mask_validos][['IDCLIENTE', 'RUT', 'RUT_CIFRADO', 'EMAIL', 'EMAIL_CIFRADO', 'ESTADO']].head(5))

Datos sensibles anonimizados.


,IDCLIENTE,RUT,RUT_CIFRADO,EMAIL,EMAIL_CIFRADO,ESTADO
502,21244,1.023.929-5,48bc2bc760ef,nyjcs326@testmail.org,95f40f4c61ef,Transformados
1858,22717,1.024.789-6,16563e4a0e0d,ihkwcbxwsg713@demo.net,bf1b34cf82e1,Transformados
3949,25171,1.035.113-0,5fa01129297f,jtkmfywd194@example.com,bc75b4fa8a79,Transformados
3093,24312,1.038.198-3,b08642dc3157,wrajwzmq353@sample.cl,5d49bfe4b01f,Transformados
4986,25561,1.047.286-2,8ecece68d5c6,elouyn277@testmail.org,d5221c4f5cda,Transformados


In [ ]:
# Creamos la copia de trabajo
cliente_procesado = df_clientes.copy()


# 1. TRATAMIENTO PREVENTIVO DE NULOS

# Reemplazamos los NaN por strings vacíos o etiquetas profesionales 
# para que las funciones de limpieza no se encuentren con valores nulos.

cliente_procesado['RUT'] = cliente_procesado['RUT'].fillna('SIN RUT')
cliente_procesado['NOMBRE'] = cliente_procesado['NOMBRE'].fillna('Nombre No Registrado')
cliente_procesado['APELLIDOPATERNO'] = cliente_procesado['APELLIDOPATERNO'].fillna('Apellido No Registrado')
cliente_procesado['APELLIDOMATERNO'] = cliente_procesado['APELLIDOMATERNO'].fillna('Apellido No Registrado')
cliente_procesado['TELEFONO'] = cliente_procesado['TELEFONO'].fillna('Sin Contacto')
cliente_procesado['EMAIL'] = cliente_procesado['EMAIL'].fillna('Sin Email')
cliente_procesado['DIRECCION'] = cliente_procesado['DIRECCION'].fillna('Dirección No Proporcionada')
cliente_procesado['CIUDAD'] = cliente_procesado['CIUDAD'].fillna('Sin Ciudad')

# ------------------------------------------------------------
# 2. FUNCIONES DE TRANSFORMACIÓN MEJORADAS
# ------------------------------------------------------------

def estandarizar_rut_pro(rut):
    if rut == 'SIN RUT': return 'RUT NO REGISTRADO'
    limpio = re.sub(r'[^0-9Kk]', '', str(rut).upper())
    return f"{limpio[:-1]}-{limpio[-1]}" if len(limpio) > 1 else 'RUT INVÁLIDO'

def cifrar_dato_pro(dato):
    # Si el dato es una de nuestras etiquetas de "Sin registro", no lo ciframos
    etiquetas_vacias = ['Sin Contacto', 'Sin Email', 'Nombre No Registrado']
    if dato in etiquetas_vacias:
        return dato
    return hashlib.sha256(str(dato).encode('utf-8')).hexdigest()

# 3. APLICACIÓN DE REGLAS DE NEGOCIO
# A) RUT
cliente_procesado['RUT_ESTANDAR'] = cliente_procesado['RUT'].apply(estandarizar_rut_pro)

# B) Nombres y Apellidos (Manejo de CSV y NaN)
# Separamos la columna NOMBRE que puede traer "Samuel, Zapata"
nombres_split = cliente_procesado['NOMBRE'].str.split(',', expand=True)

# 3. El primer pedazo SIEMPRE es el Nombre
cliente_procesado['NUEVO_NOMBRE'] = nombres_split[0].str.strip()
#Si el split dio 'None' (porque no había coma), usamos fillna para traer el dato de la columna original.
if 1 in nombres_split.columns:
    cliente_procesado['NUEVO_APE_PAT'] = nombres_split[1].str.strip().fillna(cliente_procesado['APELLIDOPATERNO'])
else:
    cliente_procesado['NUEVO_APE_PAT'] = cliente_procesado['APELLIDOPATERNO']

# 5. Apellido Materno: Misma lógica con el pedazo 3 del split.
if 2 in nombres_split.columns:
    cliente_procesado['NUEVO_APE_MAT'] = nombres_split[2].str.strip().fillna(cliente_procesado['APELLIDOMATERNO'])
else:
    cliente_procesado['NUEVO_APE_MAT'] = cliente_procesado['APELLIDOMATERNO']

# C) Fechas (Crucial: evitar NaN en fecha)
# Convertimos a fecha, los errores se vuelven NaT (Not a Time)
fechas_temp = pd.to_datetime(cliente_procesado['FECHAREGISTRO'], errors='coerce', dayfirst=True)
# Reemplazamos NaT por una fecha base profesional (1900-01-01)
cliente_procesado['FECHA_ESTANDAR'] = fechas_temp.dt.strftime('%Y-%m-%d').fillna('1900-01-01')

# D) Cifrado Ético sin NaN
cliente_procesado['TELEFONO_CIFRADO'] = cliente_procesado['TELEFONO'].apply(cifrar_dato_pro)
cliente_procesado['EMAIL_CIFRADO'] = cliente_procesado['EMAIL'].apply(cifrar_dato_pro)

# 4. REVISIÓN FINAL (Comprobación de 0 nulos)
columnas_finales = [
    'RUT_ESTANDAR', 'NUEVO_NOMBRE', 'NUEVO_APE_PAT', 'NUEVO_APE_MAT',
    'FECHA_ESTANDAR', 'TELEFONO_CIFRADO', 'EMAIL_CIFRADO', 'DIRECCION'
]

print("--- COMPROBACIÓN FINAL DE NULOS (Debe ser 0 en todo) ---")
print(cliente_procesado[columnas_finales].isnull().sum())

print("\n--- MUESTRA PROFESIONAL SIN NaN ---")
display(cliente_procesado[columnas_finales].head(10))

In [ ]:
# REPORTE FINAL DE MIGRACIÓN

total = len(df_ejercicio)
sin_cambios = len(df_ejercicio[df_ejercicio['ESTADO'] == 'Sin Cambios'])
transformados = len(df_ejercicio[df_ejercicio['ESTADO'] == 'Transformados'])
descartados = len(df_ejercicio[df_ejercicio['ESTADO'] == 'Descartado'])

print("==============================================")
print("   ESTADÍSTICAS FINALES - DATAFLOW INC.       ")
print("==============================================")
print(f"Total registros evaluados:    {total}")
print(f"Registros Sin Cambios:     {sin_cambios} ({(sin_cambios/total)*100:.1f}%)")
print(f"Registros Transformados:   {transformados} ({(transformados/total)*100:.1f}%)")
print(f"Registros Descartados:     {descartados} ({(descartados/total)*100:.1f}%)")
print("==============================================")

# Mostramos el DataFrame final listo para la carga
df_final = df_ejercicio.copy()
display(df_final.head(10))

In [ ]:
# PREPARACIÓN DE CARGA (BASE DE DESTINO)

# Filtramos para quedarnos SOLO con lo que sirve (Sin Cambios y Transformados)
df_para_subir = df_final[df_final['ESTADO'] != 'Descartado'].copy()

# Eliminamos las columnas originales que ya fueron cifradas para no subirlas por error
# (Seguridad por diseño: si ya tenemos el cifrado, no necesitamos el original en la nube)
columnas_finales = [
    'IDCLIENTE', 'RUT_CIFRADO', 'NOMBRE', 'APELLIDOPATERNO', 
    'APELLIDOMATERNO', 'TELEFONO', 'EMAIL_CIFRADO', 
    'DIRECCION', 'CIUDAD', 'CODIGOPOSTAL', 'FECHAREGISTRO', 'ESTADO'
]

df_listo_para_nube = df_para_subir[columnas_finales]

print(f"Sistema listo para migrar {len(df_listo_para_nube)} registros a la base de datos en la nube.")
display(df_listo_para_nube.head(5))

In [76]:
# EJECUCIÓN DE CARGA FÍSICA A ORACLE

try:
    cursor = conexion.cursor()
    
    # 1. Creamos la tabla de destino si no existe (con los nombres de columnas corregidos)
    # Usamos VARCHAR2(255) para los campos cifrados porque el hash es largo
    cursor.execute("""
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE CLIENTES_PROCESADOS';
            EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
        END;
    """)
    
    create_table_sql = """
    CREATE TABLE CLIENTES_PROCESADOS (
        IDCLIENTE NUMBER,
        RUT_CIFRADO VARCHAR2(255),
        NOMBRE VARCHAR2(255),
        APELLIDOPATERNO VARCHAR2(255),
        APELLIDOMATERNO VARCHAR2(255),
        TELEFONO VARCHAR2(100),
        EMAIL_CIFRADO VARCHAR2(255),
        DIRECCION VARCHAR2(255),
        CIUDAD VARCHAR2(100),
        CODIGOPOSTAL VARCHAR2(20),
        FECHAREGISTRO VARCHAR2(50),
        ESTADO VARCHAR2(50)
    )
    """
    cursor.execute(create_table_sql)
    
    # 2. Preparamos los datos del DataFrame para la inserción
    # Convertimos el DataFrame a una lista de tuplas que Oracle entiende
    datos_a_insertar = [tuple(x) for x in df_listo_para_nube.values]
    
    # 3. Inserción masiva (eficiente)
    columnas = ', '.join(df_listo_para_nube.columns)
    placeholders = ', '.join([':' + str(i+1) for i in range(len(df_listo_para_nube.columns))])
    insert_sql = f"INSERT INTO CLIENTES_PROCESADOS ({columnas}) VALUES ({placeholders})"
    
    cursor.executemany(insert_sql, datos_a_insertar)
    
    # 4. Confirmar cambios
    conexion.commit()
    print(f"¡ÉXITO! Se han insertado {len(datos_a_insertar)} registros en la tabla 'CLIENTES_PROCESADOS'.")
    cursor.close()

except Exception as e:
    print(f"Error al cargar en la base de datos: {e}")
    conexion.rollback()

¡ÉXITO! Se han insertado 4521 registros en la tabla 'CLIENTES_PROCESADOS'.


In [ ]:
# GENERACIÓN DE LOG DE ERRORES (REGISTROS DESCARTADOS)

# Filtramos solo los descartados
df_descartados = df_final[df_final['ESTADO'] == 'Descartado'].copy()

if not df_descartados.empty:
    # Guardamos en un archivo plano (CSV) para auditoría externa
    nombre_log = "log_registros_descartados.csv"
    df_descartados.to_csv(nombre_log, index=False, sep=';', encoding='utf-8-sig')
    
    print(f"Se ha generado el archivo '{nombre_log}' con {len(df_descartados)} registros que no cumplieron las reglas de negocio.")
    display(df_descartados.head(5))
else:
    print("No hubo registros descartados en este proceso.")

In [78]:
# CIERRE DE CONEXIÓN Y LIMPIEZA DE RECURSOS

try:
    if 'conexion' in locals() and conexion:
        # Es buena práctica cerrar primero el cursor si quedó abierto
        # pero aquí cerramos la conexión principal
        conexion.close()
        print("Conexión a Oracle cerrada correctamente.")
        print("Proceso de migración de DataFlow Inc. finalizado con éxito.")
except Exception as e:
    print(f"Error al cerrar la conexión: {e}")

Conexión a Oracle cerrada correctamente.
Proceso de migración de DataFlow Inc. finalizado con éxito.
